In [0]:
%pip install -U databricks-sdk databricks_vectorsearch 
%pip install --upgrade openai
dbutils.library.restartPython()

In [0]:
from config import DeployConfig

In [0]:
dbutils.widgets.text("config_path", "./config/env_variables.yml")
config_path = dbutils.widgets.get("config_path")
cfg = DeployConfig.from_yaml(config_path)

In [0]:
user_online_table_path = getattr(cfg, f"user_online_table").path
vs_index_path = getattr(cfg, f"vs_index").path
vs_index_endpoint = getattr(cfg, f"vs_index").endpoint
image_table_path = getattr(cfg, f"image_table").path
brand_table_path = getattr(cfg, f"brand_table").path
image_gen_model_path = getattr(cfg, f"image_gen_model_path").path
agent_endpoint_name = getattr(cfg, f"agent_endpoint_name")

#AGENT BUILD

In [0]:
import base64
import mlflow
import pandas as pd
import os
import requests

from databricks.sdk import WorkspaceClient
from databricks.vector_search.index import VectorSearchIndex
from databricks.vector_search.client import VectorSearchClient
from io import BytesIO
from mlflow.deployments import get_deploy_client
from mlflow.entities import SpanType
from openai import OpenAI
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

In [0]:
os.environ['CLIENT_ID'] = dbutils.secrets.get("shovakeemian-scope", "shovakeemian-sp-client-id")
os.environ['CLIENT_SECRET'] = dbutils.secrets.get("shovakeemian-scope", "shovakeemian-sp-client-secret")
os.environ['OPENAI_KEY'] = dbutils.secrets.get("shovakeemian-scope", "openai-key")

In [0]:
class PET_AGENT(mlflow.pyfunc.PythonModel):
    def __init__(self):

      # self.session=requests.Session()
      # self.oauth=None
      self.CLIENT_ID = os.environ.get("CLIENT_ID") 
      self.CLIENT_SECRET = os.environ.get("CLIENT_SECRET")
      self.api_key = os.environ.get("OPENAI_KEY")
      
      # self.online_table_url='https://80ad8fdc-e516-47cb-b709-85397e83bbf7.online-tables.cloud.databricks.com/api/2.0/workspace/1444828305810485/online/pgrest/ml_shovakeemian/user_pet_features_online'
      # self.online_table_schema='feip'

      

    def load_context(self, context):
      from transformers import CLIPProcessor, CLIPModel
      from openai import OpenAI

      self.vsc=VectorSearchClient(    
          workspace_url="https://e2-demo-field-eng.cloud.databricks.com/",
          service_principal_client_id=self.CLIENT_ID,
          service_principal_client_secret=self.CLIENT_SECRET
      )

      self.openai_client = OpenAI(api_key=self.api_key)

      self.model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14")
      self.processor= CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")
      self.brand_image_path = context.artifacts.get("brand_image")

      with open(self.brand_image_path, "rb") as f:
        self.brand_image = f.read()


    # def _ensure_openai_client(self):
    #   self.openai_client = OpenAI(api_key=self.api_key)
    

    # @mlflow.trace(name="update_oauth_token")
    # def update_oauth_token(self):
    #   url = 'https://e2-demo-field-eng.cloud.databricks.com/oidc/v1/token'
    #   data = {
    #       'grant_type': 'client_credentials',
    #       'client_id': self.CLIENT_ID,
    #       'client_secret': self.CLIENT_SECRET,
    #       'scope': 'all-apis',
    #       'authorization_details': '[{"type":"unity_catalog_permission","securable_type":"table","securable_object_name":"ml_shovakeemian.feip.user_pet_features_online","operation": "ReadOnlineView"}]'
    #   }

    #   response = requests.post(url, data=data)
    #   response.raise_for_status()
    #   self.oauth = response.json().get('access_token')
    #   return self.oauth


    # @mlflow.trace(name="user_lookup", span_type=SpanType.RETRIEVER, attributes={"table": user_online_table_path})
    # def _online_table_lookup(self, user):      
    #   headers = {
    #     "Authorization": f"Bearer {self.oauth}",
    #     "Accept-Profile": self.online_table_schema
    #     }
    #   query_params = {
    #     "select": "breed"
    #     ,"user_id": f"in.({user})"
    #     }
    #   try:
    #     response = self.session.get(self.online_table_url, headers=headers, params=query_params)
    #     cat_breed=response.json()[0]['breed']
    #   except:
    #     self.update_oauth_token()
    #     response = self.session.get(self.online_table_url, headers=headers, params=query_params)
    #     cat_breed=response.json()[0]['breed']
      
    #   cat_breed=cat_breed.replace('\u200b', '')
    #   return cat_breed


    @mlflow.trace(name="compute_text_embedding", span_type=SpanType.EMBEDDING, attributes={"model": "clip-vit-large-patch14"})
    def _get_text_embedding(self, text):
      """
      computes the text embedding for a given text.
      """
      # Want to change this to call the serving endpoint when ai_query can take params of a pyfunc.
      inputs = self.processor(text=text, return_tensors="pt", padding=True)
      text_features = self.model.get_text_features(**inputs)
      return text_features.detach().numpy().tolist()[0]


    @mlflow.trace(name="pet_image_vs_imageemb_lookup", span_type=SpanType.RETRIEVER, attributes={"model": "clip-vit-large-patch14", "vs_index": vs_index_path})
    def _vector_search_retrieval(self, query):
      text_embed_query=self._get_text_embedding(query)
      index = self.vsc.get_index(endpoint_name=vs_index_endpoint, index_name=vs_index_path)
      vs_output = index.similarity_search(columns=["id", "model_input"], query_vector=text_embed_query, num_results=3)
      return vs_output


    @mlflow.trace(name="pet_id_image_lookup", span_type=SpanType.RETRIEVER, attributes={"table": image_table_path})
    def _get_image(self, vs_output):
      """
      takes the output of vs and returns original byte array
      """
      image_base64 = [result[1] for result in vs_output['result']['data_array']][0]
      return image_base64
    
    
    @mlflow.trace(name="open ai image generation", span_type=SpanType.CHAT_MODEL, attributes={"table": image_table_path})
    def _openai_image_generation(self, pet_image, brand_image):
      # self._ensure_openai_client()
      
      pet_image_bytes=base64.b64decode(pet_image)
      brand_image_bytes=bytes(brand_image)
      prompt = """
        Create a product advertisement for pet food with the images provided. Show the pet from pet_image interacting naturally with the bag of pet food found in the petfood_brand image. Make it as realistic as possible. Keep the pet in the image's original environemnt and try and incorporate the background. If there are multiple pets in the pet_image, make sure to include both; but if there is just one pet in the image, do not generate an image of a second one. Do not add any text to the ad except for the text on the bag of pet food. 
      """
      result = self.openai_client.images.edit(
          model="gpt-image-1",
          image=[("pet_image.png", pet_image_bytes, "image/jpeg"), ("petfood_brand.png", brand_image_bytes, "image/png")],
          prompt=prompt
      )
      image_base64 = result.data[0].b64_json
      return (image_base64)


    @mlflow.trace(name="ad-image-agent")
    def predict(self, context, model_input, params):

      openai_toggle=params.get('openai_toggle')

      if isinstance(model_input, pd.DataFrame):
        query = model_input["model_input"].iloc[0]
      elif isinstance(model_input, dict):
        query = model_input.get("model_input", "")
      elif isinstance(model_input, str):
        query = model_input
      else:
        raise ValueError("Unsupported input type")

      if openai_toggle:
        vs_output=self._vector_search_retrieval(query=query)
        pet_image=self._get_image(vs_output)
        final_ad=self._openai_image_generation(pet_image=pet_image, brand_image=self.brand_image)
        return query, pet_image, final_ad
      
      elif openai_toggle==False:
        vs_output=self._vector_search_retrieval(query=query)
        pet_image=self._get_image(vs_output)
        return query, pet_image, pet_image

In [0]:
agent=PET_AGENT()

class DummyContext:
    artifacts = {
        "brand_image": "/Volumes/ml_shovakeemian/feip/petfood_brand_creatives/BricksV1.png"
    }
agent.load_context(DummyContext)

In [0]:
model_input, pet_image, final_ad=agent.predict(model_input='white poodle', context=None, params={'openai_toggle': True})

In [0]:
print(model_input)

In [0]:
Image.open(BytesIO(base64.b64decode(pet_image)))

In [0]:
Image.open(BytesIO(base64.b64decode(final_ad)))

## DEPLOY

In [0]:
import pandas as pd

example_input = {
    # Your model's input data (matches input_schema)
    "inputs": ["black forest cat"],
    "params": [{"openai_toggle": False}]
}

In [0]:
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec, ParamSchema, ParamSpec

input_schema = Schema([
    ColSpec("string", "model_input")
])

output_schema = Schema([
    ColSpec("string", "model_input"),
    ColSpec("string", "pet_image"),
    ColSpec("string", "final_ad"),
])


params_schema = ParamSchema([
    ParamSpec("openai_toggle", "boolean", True)
])

signature = ModelSignature(inputs=input_schema, outputs=output_schema, params=params_schema)

In [0]:
pip_requirements=[
  "--extra-index-url https://download.pytorch.org/whl/cu121", 
  "openai==1.79.0",
  "databricks-vectorsearch==0.56",
  "mlflow==2.15.1",
  "setuptools<70.0.0", 
  "torch==2.3.1+cu121", 
  "accelerate==0.31.0", 
  "astunparse==1.6.3", 
  "bcrypt==3.2.0", 
  "boto3==1.34.39", 
  "configparser==5.2.0", 
  "defusedxml==0.7.1", 
  "dill==0.3.6",
   "google-cloud-storage==2.10.0", 
   "ipython==8.15.0", 
   "lz4==4.3.2", 
   "nvidia-ml-py==12.555.43", 
   "optree==0.12.1", 
   "pandas==1.5.3", 
   "pyopenssl==23.2.0", 
   "pytesseract==0.3.10", 
   "scikit-learn==1.3.0", 
   "sentencepiece==0.1.99", 
   "torchvision==0.18.1+cu121", 
   "transformers==4.41.2",
   "https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.4.post1/flash_attn-2.7.4.post1+cu12torch2.3cxx11abiFALSE-cp311-cp311-linux_x86_64.whl"
   ]

In [0]:
from mlflow.models.resources import DatabricksVectorSearchIndex

resources = [DatabricksVectorSearchIndex(index_name=vs_index_path)]

with mlflow.start_run():
  mlflow.pyfunc.log_model(
    "agent", 
    python_model=PET_AGENT(),
    resources=resources,
    signature=signature,
    pip_requirements=pip_requirements,
    input_example=example_input,
    artifacts={
      "brand_image": "/Volumes/ml_shovakeemian/feip/petfood_brand_creatives/BricksV1.png"},
  )

In [0]:
run_id = mlflow.last_active_run().info.run_id

In [0]:
mlflow.set_registry_uri("databricks-uc")

# Register the model to UC
uc_registered_model_info = mlflow.register_model(
    model_uri=f"runs:/{run_id}/agent", name=image_gen_model_path
)

In [0]:
test_loaded=mlflow.pyfunc.load_model(f"models:/{image_gen_model_path}/10")

In [0]:
model_input, pet_image, final_ad = test_loaded.predict(
    {'model_input': 'black forest cat'},
    params={'openai_toggle': False}
)

In [0]:
Image.open(BytesIO(base64.b64decode(pet_image)))

In [0]:
Image.open(BytesIO(base64.b64decode(final_ad)))

In [0]:
from mlflow.deployments import get_deploy_client

client = get_deploy_client("databricks")
# response = client.create_endpoint(
#     name=agent_endpoint_name,
#     config={
#         "served_entities": [
#             {
#                 "name": agent_endpoint_name,
#                 "entity_name": image_gen_model_path,
#                 "entity_version": uc_registered_model_info.version,
#                 "workload_size": "Small",
#                 "scale_to_zero_enabled": True,
#                 "workload_type": "GPU_SMALL",
#                 "environment_vars": {
#                     "CLIENT_ID": "{{secrets/shovakeemian-scope/shovakeemian-sp-client-id}}",
#                     "CLIENT_SECRET": "{{secrets/shovakeemian-scope/shovakeemian-sp-client-secret}}",
#                     "OPENAI_KEY": "{{secrets/shovakeemian-scope/openai-key}}",
#                 }
#             }
#         ],
#     }
# )

# Update the existing endpoint
response = client.update_endpoint(
    endpoint=agent_endpoint_name, # existing endpoint name
    config={
        "served_entities": [
            {
                "name": agent_endpoint_name,
                "entity_name": image_gen_model_path,
                "entity_version": uc_registered_model_info.version,
                "workload_type": "GPU_SMALL",
                "workload_size": "Small",
                "scale_to_zero_enabled": True,
                "environment_vars": {
                    "CLIENT_ID": "{{secrets/shovakeemian-scope/shovakeemian-sp-client-id}}",
                    "CLIENT_SECRET": "{{secrets/shovakeemian-scope/shovakeemian-sp-client-secret}}",
                    "OPENAI_KEY": "{{secrets/shovakeemian-scope/openai-key}}",
                }
            }
        ]
    }
)

print(response)

In [0]:
# to query endpoint:
{
    "inputs": [
      "siamese kitten"
    ],
    "params": 
      {
        "openai_toggle": false
      }
}